# **Mapping Topik BERTopic -> 15 Aspek (rujuk FiQA)**

Memetakan 46 topik BERTopic ke **maksimal 15 aspek**, dikelompokkan di bawah 4 kategori FiQA: `Corporate`, `Stock`, `Market`, `Economy` (Maia et al. 2018; Lengkeek et al. 2023). Topik semakna digabung sehingga jumlah aspek ringkas.

- Input: `data/processed/sentences_with_topic.csv` + `data/processed/topic_info.csv`.
- Output: `data/processed/mapping_aspek.csv` (tabel cek per topik) + `data/processed/sentences_with_aspect.csv` (kalimat + aspek).

Mapping ini **usulan awal berbasis keyword** - keputusan final tetap manual. Topik `-1` (outlier) & topik tak jelas -> **Lainnya** (bukan salah satu dari 15 aspek).

In [1]:
from pathlib import Path
import pandas as pd

sent = pd.read_csv("data/processed/sentences_with_topic.csv", dtype={"id_artikel": str})
info = pd.read_csv("data/processed/topic_info.csv")
print(f"Kalimat: {len(sent):,} | Topik: {len(info)}")

Kalimat: 63,632 | Topik: 47


## 15 aspek final + pemetaan tiap topik

`ASPEK` mendefinisikan 15 aspek (nama + kategori FiQA). `TOPIC2ASPEK` memetakan tiap topik ke salah satu aspek, atau ke `Lainnya`. Edit bebas setelah membaca kalimat representatif.

In [2]:
# 15 ASPEK FINAL: nama -> kategori FiQA
ASPEK = {
    # CORPORATE (6)
    "Kinerja Keuangan & Laba":            "Corporate",
    "Kredit & Kualitas Aset":             "Corporate",
    "Dividen & Aksi Korporasi":           "Corporate",
    "Manajemen & Kepemimpinan":           "Corporate",
    "Layanan & Produk":                   "Corporate",
    "Strategi & Ekspansi Bisnis":         "Corporate",
    # STOCK (4)
    "Pergerakan Saham Bank":              "Stock",
    "Aksi Investor Asing":                "Stock",
    "Saham Spesifik Non-Bank":            "Stock",
    "Valuasi & Kapitalisasi":             "Stock",
    # MARKET (3)
    "Pergerakan IHSG & Indeks":           "Market",
    "Sektor & Transaksi Pasar":           "Market",
    "Bursa Global":                       "Market",
    # ECONOMY (2)
    "Kebijakan Moneter & Nilai Tukar":    "Economy",
    "Makroekonomi & Faktor Global":       "Economy",
}
assert len(ASPEK) == 15, f"harus 15 aspek, sekarang {len(ASPEK)}"

# pemetaan tiap topik -> nama aspek (atau 'Lainnya')
TOPIC2ASPEK = {
    -1: "Lainnya", 11: "Lainnya", 17: "Lainnya", 39: "Lainnya",
    # Corporate
    14: "Kinerja Keuangan & Laba",
    13: "Kredit & Kualitas Aset", 23: "Kredit & Kualitas Aset", 34: "Kredit & Kualitas Aset", 36: "Kredit & Kualitas Aset",
    7:  "Dividen & Aksi Korporasi", 21: "Dividen & Aksi Korporasi", 26: "Dividen & Aksi Korporasi",
    16: "Manajemen & Kepemimpinan",
    10: "Layanan & Produk", 22: "Layanan & Produk",
    2:  "Strategi & Ekspansi Bisnis", 9: "Strategi & Ekspansi Bisnis", 19: "Strategi & Ekspansi Bisnis",
    # Stock
    1:  "Pergerakan Saham Bank", 32: "Pergerakan Saham Bank", 44: "Pergerakan Saham Bank",
    5:  "Aksi Investor Asing", 8: "Aksi Investor Asing", 20: "Aksi Investor Asing", 35: "Aksi Investor Asing",
    31: "Saham Spesifik Non-Bank", 33: "Saham Spesifik Non-Bank", 45: "Saham Spesifik Non-Bank",
    40: "Valuasi & Kapitalisasi",
    # Market
    0:  "Pergerakan IHSG & Indeks", 12: "Pergerakan IHSG & Indeks", 15: "Pergerakan IHSG & Indeks",
    4:  "Sektor & Transaksi Pasar", 6: "Sektor & Transaksi Pasar", 37: "Sektor & Transaksi Pasar",
    25: "Bursa Global", 38: "Bursa Global", 41: "Bursa Global",
    # Economy
    3:  "Kebijakan Moneter & Nilai Tukar", 18: "Kebijakan Moneter & Nilai Tukar", 24: "Kebijakan Moneter & Nilai Tukar",
    27: "Kebijakan Moneter & Nilai Tukar", 28: "Kebijakan Moneter & Nilai Tukar",
    29: "Makroekonomi & Faktor Global", 30: "Makroekonomi & Faktor Global", 42: "Makroekonomi & Faktor Global", 43: "Makroekonomi & Faktor Global",
}

# cek: semua topik tercakup
topik_data = set(int(t) for t in info["Topic"])
belum = sorted(topik_data - set(TOPIC2ASPEK))
print("Topik belum dipetakan:", belum if belum else "tidak ada (semua tercakup)")
print(f"Jumlah aspek: {len(ASPEK)} (+ Lainnya)")

Topik belum dipetakan: tidak ada (semua tercakup)
Jumlah aspek: 15 (+ Lainnya)


## Tabel mapping per topik (untuk dicek manual)

In [3]:
def aspek_of(t):
    return TOPIC2ASPEK.get(int(t), "Lainnya")
def kategori_of(a):
    return ASPEK.get(a, "Lainnya")

info["nama_aspek"]   = info["Topic"].map(aspek_of)
info["kategori_fiqa"] = info["nama_aspek"].map(kategori_of)

mapping = info[["Topic", "Count", "kategori_fiqa", "nama_aspek", "Keywords",
                "Representative_1", "Representative_2", "Representative_3"]].copy()
mapping = mapping.sort_values(["kategori_fiqa", "nama_aspek", "Topic"]).reset_index(drop=True)
mapping

,Topic,Count,kategori_fiqa,nama_aspek,Keywords,Representative_1,Representative_2,Representative_3
0,7,1450,Corporate,Dividen & Aksi Korporasi,"2026, 2025, dividen, tanggal, rapat, dijadwalk...",Adapun cum date dividen di pasar reguler dan n...,Keputusan tersebut telah memperoleh persetujua...,Perseroan menetapkan tanggal cum dividen pada ...
1,21,648,Corporate,Dividen & Aksi Korporasi,"per saham, per, dividen, saham, interim, tahun...",Lalu ada juga Batavia Prosperindo Internasiona...,"Sebelumnya, perseroan telah membagikan dividen...","Sementara itu, PT Bank Mandiri (Persero) Tbk (..."
2,26,463,Corporate,Dividen & Aksi Korporasi,"buyback, pembelian saham, pembelian, pelaksana...",Rapat Umum Pemegang Saham (RPUS) PT Bank Negar...,Periode pelaksanaan buyback dilakukan dalam wa...,Persetujuan atas rencana pembelian kembali sah...
3,14,1112,Corporate,Kinerja Keuangan & Laba,"pendapatan, laba, meningkat, sebesar, laba ber...",Perseroan menargetkan pendapatan tahun 2026 me...,"Laba bersih perseroan tumbuh 16,40% secara tah...","Alhasil, laba bersih yang diatribusikan kepada..."
4,13,1122,Corporate,Kredit & Kualitas Aset,"kredit, bank, bank mandiri, mandiri, perbankan...","""Fokus penyaluran kredit akan diarahkan kepada...",Direktur Utama Bank Mandiri Riduan mengungkapk...,"Oleh sebab itu, kami akan terus menjaga pertum..."
5,23,563,Corporate,Kredit & Kualitas Aset,"kredit, tumbuh, pertumbuhan, year, pertumbuhan...",Pertumbuhan kredit diikuti dengan pertumbuhan ...,Pertumbuhan kredit ditopang oleh dana pihak ke...,Kredit industri perbankan hingga Triwulan II-2...
6,34,378,Corporate,Kredit & Kualitas Aset,"bri, rp1, tumbuh, total, sebesar, debitur, pen...","Hingga Triwulan II 2026, total aset BRI Group ...","Hingga kuartal II-2026, total aset BRI Group m...","Hingga akhir Triwulan I 2026, BRI berhasil men..."
7,36,310,Corporate,Kredit & Kualitas Aset,"npl, loan, loan npl, rasio, non performing, pe...",Rasio kredit bermasalah atau non performing lo...,"Pada saat yang sama, kualitas aset terus memba...",Rasio kredit bermasalah atau non-performing lo...
8,10,1210,Corporate,Layanan & Produk,"digital, layanan, bni, kartu, nasabah, pembaya...","Di samping layanan operasional terbatas, BNI j...","Selain itu, masyarakat diimbau dapat memanfaat...",Nasabah tetap dapat dengan mudah mengakses lay...
9,22,622,Corporate,Layanan & Produk,"rekening, saldo, nasabah, minimum, bank, tabun...",Perbankan memiliki kebijakan yang berbeda satu...,Saldo minimum adalah jumlah dana yang wajib me...,Bagi nasabah bank perlu mengetahui saldo minim...


## Ringkasan: distribusi per aspek & per kategori

In [4]:
tot = info["Count"].sum()
print("Distribusi kalimat per ASPEK:")
for a, v in info.groupby("nama_aspek")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {a:34} {v:>6,} ({v/tot*100:4.1f}%)")
print("\nDistribusi kalimat per KATEGORI FiQA:")
for k, v in info.groupby("kategori_fiqa")["Count"].sum().sort_values(ascending=False).items():
    print(f"  {k:12} {v:>6,} ({v/tot*100:4.1f}%)")

Distribusi kalimat per ASPEK:
  Lainnya                            24,071 (37.8%)
  Pergerakan IHSG & Indeks            7,107 (11.2%)
  Kebijakan Moneter & Nilai Tukar     4,326 ( 6.8%)
  Strategi & Ekspansi Bisnis          4,178 ( 6.6%)
  Aksi Investor Asing                 4,001 ( 6.3%)
  Sektor & Transaksi Pasar            3,616 ( 5.7%)
  Pergerakan Saham Bank               3,494 ( 5.5%)
  Dividen & Aksi Korporasi            2,561 ( 4.0%)
  Kredit & Kualitas Aset              2,373 ( 3.7%)
  Layanan & Produk                    1,832 ( 2.9%)
  Makroekonomi & Faktor Global        1,451 ( 2.3%)
  Kinerja Keuangan & Laba             1,112 ( 1.7%)
  Bursa Global                        1,089 ( 1.7%)
  Saham Spesifik Non-Bank             1,089 ( 1.7%)
  Manajemen & Kepemimpinan            1,039 ( 1.6%)
  Valuasi & Kapitalisasi                293 ( 0.5%)

Distribusi kalimat per KATEGORI FiQA:
  Lainnya      24,071 (37.8%)
  Corporate    13,095 (20.6%)
  Market       11,812 (18.6%)
  Stock  

## Terapkan ke tiap kalimat & simpan 2 output

In [5]:
sent["nama_aspek"]   = sent["topic"].map(aspek_of)
sent["kategori_fiqa"] = sent["nama_aspek"].map(kategori_of)

OUT = Path("data/processed")
mapping.to_csv(OUT / "mapping_aspek.csv", index=False, encoding="utf-8-sig")
sent.to_csv(OUT / "sentences_with_aspect.csv", index=False, encoding="utf-8-sig")
print(f"[1] mapping_aspek.csv -> {len(mapping)} topik, {info['nama_aspek'].nunique()-1} aspek + Lainnya")
print(f"[2] sentences_with_aspect.csv -> {len(sent):,} kalimat")

[1] mapping_aspek.csv -> 47 topik, 15 aspek + Lainnya
[2] sentences_with_aspect.csv -> 63,632 kalimat


## Langkah manual berikutnya

1. Buka `mapping_aspek.csv`, baca `Keywords` + 3 kalimat representatif, **koreksi** penempatan topik ke aspek bila perlu.
2. Tulis **definisi 1 kalimat + contoh batas** tiap aspek di `daftar_aspek.md`.
3. **Validasi penetapan aspek pada sampel**, lalu **kunci** daftar aspek sebelum anotasi sentimen (Fase 04).